In [12]:
import pandas as pd
import numpy as np

# =========================
# 1. Load data
# =========================
df = pd.read_csv("data1C/model_option_A.csv")

# Parse dates
df["date"] = pd.to_datetime(df["date"])

# Sort by user and date
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# =========================
# 2. Make sure mood_next_day exists
#    (Option A should already contain it, but this is a safe fallback)
# =========================
if "mood_next_day" not in df.columns:
    df["mood_next_day"] = df.groupby("id")["mood"].shift(-1)

# =========================
# 3. Current-day features to keep
# =========================
current_day_features = [
    "mood",
    "activity",
    "screen",
    "call",
    "sms",
    "appCat.builtin",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.finance",
    "appCat.game",
    "appCat.office",
    "appCat.other",
    "appCat.social",
    "appCat.travel",
    "appCat.unknown",
    "appCat.utilities",
    "appCat.weather",
    "circumplex.arousal",
    "circumplex.valence",
]

# =========================
# 4. Add lag1 features only
# =========================
lag1_features = ["mood", "activity", "screen"]

for col in lag1_features:
    df[f"{col}_lag1"] = df.groupby("id")[col].shift(1)

# =========================
# 5. Add 3-day mean history features
#    History uses previous days only, so shift(1) first
# =========================
mean_3d_features = [
    "mood",
    "activity",
    "screen",
    "call",
    "sms",
    "appCat.social",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.other",
    "appCat.utilities"
]

for col in mean_3d_features:
    shifted = df.groupby("id")[col].shift(1)
    df[f"{col}_mean_3d_hist"] = (
        shifted.groupby(df["id"]).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    )

# =========================
# 6. Add 3-day std only for key variables
# =========================
std_3d_features = ["mood", "activity", "screen"]

for col in std_3d_features:
    shifted = df.groupby("id")[col].shift(1)
    df[f"{col}_std_3d_hist"] = (
        shifted.groupby(df["id"]).rolling(3, min_periods=2).std().reset_index(level=0, drop=True)
    )

# =========================
# 7. Add change features
# =========================
df["mood_change_today_vs_yesterday"] = df["mood"] - df["mood_lag1"]
df["activity_change_today_vs_yesterday"] = df["activity"] - df["activity_lag1"]
df["screen_change_today_vs_yesterday"] = df["screen"] - df["screen_lag1"]

# =========================
# 8. Add calendar features
# =========================
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# =========================
# 9. Drop rows without enough history
#    Require lag1 and next-day target
# =========================
required_columns = [
    "mood_lag1",
    "activity_lag1",
    "screen_lag1",
    "mood_next_day"
]

df_final = df.dropna(subset=required_columns).copy()

# =========================
# 10. Keep only useful columns
# =========================
final_columns = (
    ["id", "date"] +
    current_day_features +
    [f"{col}_lag1" for col in lag1_features] +
    [f"{col}_mean_3d_hist" for col in mean_3d_features] +
    [f"{col}_std_3d_hist" for col in std_3d_features] +
    [
        "mood_change_today_vs_yesterday",
        "activity_change_today_vs_yesterday",
        "screen_change_today_vs_yesterday",
        "day_of_week",
        "is_weekend",
        "mood_next_day"
    ]
)

# Only keep columns that actually exist
final_columns = [col for col in final_columns if col in df_final.columns]

df_final = df_final[final_columns]

# =========================
# 11. Save final dataset
# =========================
df_final.to_csv("data1C/previous_final_data_1C.csv", index=False)

print("Saved file: data1C/previous_final_data_1C.csv")
print("Shape:", df_final.shape)
print(df_final.head())
print(df_final.columns.tolist())

Saved file: data1C/previous_final_data_1C.csv
Shape: (1090, 43)
        id       date  mood  activity        screen  call  sms  \
2  AS14.01 2014-03-22  6.40  0.236880   6142.161000   3.0  1.0   
3  AS14.01 2014-03-23  6.80  0.142741   6773.832001   0.0  0.0   
4  AS14.01 2014-03-24  6.00  0.078961  15047.351001  10.0  0.0   
5  AS14.01 2014-03-25  6.75  0.098374  15381.207999   0.0  1.0   
6  AS14.01 2014-03-26  6.60  0.101308  13585.707000   0.0  0.0   

   appCat.builtin  appCat.communication  appCat.entertainment  ...  \
2         731.429              4962.918                93.324  ...   
3        1286.246              5237.319                94.346  ...   
4         866.956              9270.629               976.971  ...   
5        1032.768             10276.751                68.206  ...   
6        1167.497              8988.753               910.479  ...   

   appCat.utilities_mean_3d_hist  mood_std_3d_hist  activity_std_3d_hist  \
2                     299.377000          

In [13]:
import pandas as pd
import numpy as np

# =========================
# 1. Load final dataset
# =========================
df = pd.read_csv("data1C/previous_final_data_1C.csv", na_values=["", " "])

# Clean column names just in case
df.columns = df.columns.str.strip()

# Parse date
df["date"] = pd.to_datetime(df["date"])

# Sort properly
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# =========================
# 2. Define feature groups
# =========================

# Self-reported continuous variables:
# Impute with within-person linear interpolation
self_report_cols = [
    "mood",
    "activity",
    "circumplex.arousal",
    "circumplex.valence"
]

# Usage / count / event variables:
# Impute with 0 on the daily level
zero_fill_cols = [
    "screen",
    "call",
    "sms",
    "appCat.builtin",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.finance",
    "appCat.game",
    "appCat.office",
    "appCat.other",
    "appCat.social",
    "appCat.travel",
    "appCat.unknown",
    "appCat.utilities",
    "appCat.weather"
]

# Keep only columns that actually exist
self_report_cols = [col for col in self_report_cols if col in df.columns]
zero_fill_cols = [col for col in zero_fill_cols if col in df.columns]

# =========================
# 3. Impute self-reported features within each person
# =========================
for col in self_report_cols:
    df[col] = (
        df.groupby("id")[col]
        .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
    )

# =========================
# 4. Fill usage/event features with 0
# =========================
for col in zero_fill_cols:
    df[col] = df[col].fillna(0)

# =========================
# 5. Recompute engineered features
# =========================

# Lag-1 features
lag1_features = ["mood", "activity", "screen"]
lag1_features = [col for col in lag1_features if col in df.columns]

for col in lag1_features:
    df[f"{col}_lag1"] = df.groupby("id")[col].shift(1)

# 3-day historical mean features
mean_3d_features = [
    "mood",
    "activity",
    "screen",
    "call",
    "sms",
    "appCat.social",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.other",
    "appCat.utilities"
]
mean_3d_features = [col for col in mean_3d_features if col in df.columns]

for col in mean_3d_features:
    shifted = df.groupby("id")[col].shift(1)
    df[f"{col}_mean_3d_hist"] = (
        shifted.groupby(df["id"])
        .rolling(3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

# 3-day historical std features
std_3d_features = ["mood", "activity", "screen"]
std_3d_features = [col for col in std_3d_features if col in df.columns]

for col in std_3d_features:
    shifted = df.groupby("id")[col].shift(1)
    df[f"{col}_std_3d_hist"] = (
        shifted.groupby(df["id"])
        .rolling(3, min_periods=2)
        .std()
        .reset_index(level=0, drop=True)
    )

# Replace std NaN (not enough history) with 0
for col in std_3d_features:
    std_col = f"{col}_std_3d_hist"
    if std_col in df.columns:
        df[std_col] = df[std_col].fillna(0)

# Change features
if "mood" in df.columns and "mood_lag1" in df.columns:
    df["mood_change_today_vs_yesterday"] = df["mood"] - df["mood_lag1"]

if "activity" in df.columns and "activity_lag1" in df.columns:
    df["activity_change_today_vs_yesterday"] = df["activity"] - df["activity_lag1"]

if "screen" in df.columns and "screen_lag1" in df.columns:
    df["screen_change_today_vs_yesterday"] = df["screen"] - df["screen_lag1"]

# Calendar features
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# =========================
# 6. Drop unusable rows
# =========================
required_columns = [
    "mood_lag1",
    "activity_lag1",
    "screen_lag1",
    "mood_next_day"
]
required_columns = [col for col in required_columns if col in df.columns]

df_imputed = df.dropna(subset=required_columns).copy()

# =========================
# 7. Save result
# =========================
df_imputed.to_csv("data1C/final_data_1C.csv", index=False)

# =========================
# 8. Missing value overview after imputation
# =========================
missing_overview = pd.DataFrame({
    "missing_count": df_imputed.isna().sum(),
    "missing_percentage": df_imputed.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

print("Saved file: data1C/final_data_1C.csv")
print("Shape:", df_imputed.shape)
print("\nMissing values after imputation:")
print(missing_overview)
print("\nFirst rows:")
print(df_imputed.head())

Saved file: data1C/final_data_1C.csv
Shape: (1063, 43)

Missing values after imputation:
                                    missing_count  missing_percentage
id                                              0                 0.0
appCat.other_mean_3d_hist                       0                 0.0
mood_mean_3d_hist                               0                 0.0
activity_mean_3d_hist                           0                 0.0
screen_mean_3d_hist                             0                 0.0
call_mean_3d_hist                               0                 0.0
sms_mean_3d_hist                                0                 0.0
appCat.social_mean_3d_hist                      0                 0.0
appCat.communication_mean_3d_hist               0                 0.0
appCat.entertainment_mean_3d_hist               0                 0.0
appCat.utilities_mean_3d_hist                   0                 0.0
activity_lag1                                   0                 0.0
m